# Seminar Notebook 3.1: Making a DFM for Week 7 Seminar

**LSE MY459: Computational Text Analysis and Large Language Models** (WT 2026)

**Ryan Hübert**

This notebook creates a document-feature matrix (DFM) from the candidate tweets dataset that we will use in subsequent notebooks this week.

## Directory management

We begin with some directory management to specify the file path to the folder on your computer where you wish to store data for this notebook.

In [ ]:
# Directory for our seminar materials this week
import os
sdir = os.path.join(os.path.expanduser("~"), "LSE-MY459-WT26", "SeminarWeek07") # or whatever path you want
if not os.path.exists(sdir):
    os.makedirs(sdir)

# Subfolder for raw downloaded data
rdir = os.path.join(sdir, "raw")
if not os.path.exists(rdir):
    os.makedirs(rdir)

## Candidate Tweets

Our corpus contains tweets posted by the major candidates for the 2016 U.S. presidential election. We will download this from the course GitHub data repository.

In [ ]:
# Where is the remote file?
rfile_candidate = "https://raw.githubusercontent.com/lse-my459/data/main/candidate-tweets.csv"

# Where will we store the local copy of it?
lfile_candidate = os.path.join(rdir, "candidate-tweets.csv")

# Check if you have the file yet and if not, download it to correct location
if not os.path.exists(lfile_candidate):
    import requests
    r = requests.get(rfile_candidate)
    r.raise_for_status()
    with open(lfile_candidate, "wb") as f:
        f.write(r.content)
    print(f"Downloaded {lfile_candidate}")
else:
    print(f"File already exists: {lfile_candidate}")

Let's load the dataframe, and get a sense for what it contains. First, we load and look at the first few rows.

In [ ]:
import pandas as pd
candidate = pd.read_csv(lfile_candidate, dtype="object")
candidate.head()

Next, let's see whose tweets appear in this corpus.

In [ ]:
candidate.groupby("screen_name")['text'].count()

We can see that this contains tweets by Democratic Party candidates [Bernie Sanders](https://en.wikipedia.org/wiki/Bernie_Sanders_2016_presidential_campaign) and [Hillary Clinton](https://en.wikipedia.org/wiki/Hillary_Clinton_2016_presidential_campaign), as well as Republican Party candidates [Ted Cruz](https://en.wikipedia.org/wiki/Ted_Cruz_2016_presidential_campaign) and [Donald Trump](https://en.wikipedia.org/wiki/Donald_Trump_2016_presidential_campaign).

### Preprocessing

Since our corpus contains tweets, we'll use a standard set of preprocessing steps. We will define a function below that takes text (a `str` object) and returns a preprocessed list of tokens (unigrams). We will add some flexibility allowing us to decide whether to remove hashtags and/or user handles. In principle, we could add additional arguments allowing us to decide whether to remove stop words, stem, etc. 

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.stem import snowball

sw = [x.lower() for x in stopwords.words('english')]
sstemmer = snowball.SnowballStemmer("english")

def preprocess_text(text, remove_hashtags = False, remove_handles = False):    
    if pd.isna(text):
        return []
    text = str(text)
    # Remove escaped Unicode codepoints like <U+2019> that appear in some datasets
    text = re.sub(r"<[Uu]\+[0-9A-Fa-f]+>", "", text)
    # Tokenise on whitespace or hyphens
    tokens = re.split(r"[\s\-]+", text)
    # Lowercase
    tokens = [x.lower() for x in tokens]
    # Remove URLs
    tokens = [x for x in tokens if not re.search(r"^https?://", x)]
    # Remove HTML character entities
    tokens = [x for x in tokens if not re.search(r"^&.+;$", x)]
    # Remove all extra characters
    tokens = [re.sub(r"[^A-Za-z0-9\-\''@\# ]", "", x) for x in tokens]
    # Remove ending quotes
    tokens = [re.sub(r"[\'']$", "", x) for x in tokens]
    # Keep only tokens that begin with a letter, at or hash
    regex = r"^[A-Za-z"
    regex = regex + r"\#" if not remove_hashtags else regex
    regex = regex + r"@" if not remove_handles else regex
    regex = re.compile(regex + r"]")
    tokens = [x for x in tokens if re.search(regex, x)]
    # Remove stopwords
    tokens = [x for x in tokens if x not in sw]
    # Stem using snowball stemmer
    tokens = [sstemmer.stem(x) for x in tokens]
    # Drop empty strings, the word "rt" and single character strings
    tokens = [x for x in tokens if x != "rt" and len(x) > 1]
    return tokens

Next, we apply the preprocessing function and count the tokens corresponding to each tweet in the corpus.

In [ ]:
from collections import Counter

candidate["preprocessed"] = candidate["text"].apply(lambda x: preprocess_text(x, True, True))
candidate["term_freqs"] = candidate["preprocessed"].map(Counter)
candidate[["text", "preprocessed", "term_freqs"]].head()

Now, we create a DFM (as a sparse array).

In [ ]:
from sklearn.feature_extraction import DictVectorizer

dv_candidate = DictVectorizer()
dfm_candidate = dv_candidate.fit_transform(candidate["term_freqs"].to_list())
vocabulary_candidate = dv_candidate.get_feature_names_out()
print(dfm_candidate.shape) 

Notice that this DFM has 11,878 columns. To improve our computations, we will trim this corpus by keeping any types in the vocabulary (i.e., columns) that meet two criteria:

- **High term frequency**: the type is in the top 20% of most used words.
- **Low document frequency**: the type is used in no more than 10% of documents.

In [ ]:
import numpy as np

# Step 1: calculate TTF and DF for each feature
ttf_candidate = dfm_candidate.sum(axis=0).A1
docf_candidate = (dfm_candidate > 0).sum(axis=0).A1

# Step 2: define cutoffs
ttf_candidate_min = np.quantile(ttf_candidate, 0.80)
docf_candidate_max = dfm_candidate.shape[0] * 0.1

# Step 3: trim the DFM and the vocabulary
keep_candidate = (ttf_candidate >= ttf_candidate_min) & (docf_candidate <= docf_candidate_max)
dfm_candidate = dfm_candidate[:, keep_candidate]
vocabulary_candidate = vocabulary_candidate[keep_candidate]

print(dfm_candidate.shape)

Since we will use this DFM for classification, we should remove any features that contain the candidates' names. These would be trivially predictive of the author and would prevent the classifier from learning anything interesting about writing style or content. We stem the names using the same stemmer, and then drop any feature that contains any of the stemmed name substrings.

In [ ]:
# Remove features containing candidate names
name_parts = ["donald", "trump", "hillary", "clinton", "ted", "cruz", "bernie", "sanders"]
name_stems = name_parts + list(set([sstemmer.stem(x) for x in name_parts]))
name_mask = np.array([not any(s in v for s in name_stems) for v in vocabulary_candidate])
print(vocabulary_candidate[~name_mask].tolist()) # print removed features

dfm_candidate = dfm_candidate[:, name_mask]
vocabulary_candidate = vocabulary_candidate[name_mask]
print(dfm_candidate.shape)

This is now a lot more manageable, since it only has 2,395 columns. Now let's quickly examine the top features in a word cloud.

In [ ]:
import wordcloud
import matplotlib.pyplot as plt

# Count top features
top_features_candidate = dfm_candidate.sum(axis=0).A1
top_features_candidate = pd.Series(top_features_candidate, index=vocabulary_candidate).astype(int)

# Initialise the wordcloud object as `wc`
wc = wordcloud.WordCloud(width=800,
                         height=400,
                         background_color="white",
                         relative_scaling = 1,
                         max_words=200, 
                         random_state=42)

# Feed in the data from our `top_features` object
wc = wc.generate_from_frequencies(top_features_candidate)

# Don't use the default colours, and make blue
wc = wc.recolor(color_func=lambda *args, **kwargs: (51,51,255))

plt.figure(figsize=(8, 4))
plt.imshow(wc)
plt.axis("off")
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.show()

Finally, let's save the DFM and corpus data.

In [ ]:
from scipy import sparse

# Save the sparse DFM
sparse.save_npz(os.path.join(sdir, "candidate-dfm.npz"), dfm_candidate)

# Save the vocabulary
with open(os.path.join(sdir, "candidate-dfm-features.txt"), "w") as f:
    f.write("\n".join(vocabulary_candidate))

# Save the corpus metadata (excluding preprocessed columns)
candidate_cols = [c for c in candidate.columns if c not in ["preprocessed", "term_freqs"]]
candidate[candidate_cols].to_csv(os.path.join(sdir, "candidate-corpus.csv"), index=False)